In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("¡Librerías cargadas correctamente!")
print("Versión de Pandas:", pd.__version__)

Matplotlib is building the font cache; this may take a moment.


¡Librerías cargadas correctamente!
Versión de Pandas: 3.1.0


In [2]:
from pathlib import Path

ruta = Path("data/raw/operaciones-aeropuertos.csv")

if not ruta.exists():
    ruta = Path("../data/raw/operaciones-aeropuertos.csv")

df = pd.read_csv(ruta)

print("Cantidad de filas:", df.shape[0])
print("Cantidad de columnas:", df.shape[1])

df.head()

Cantidad de filas: 16172
Cantidad de columnas: 4


,mes_id,aeropuerto_oaci,internacional_domestico,cnt_operaciones
0,200001,SCAP,D,6
1,200001,SCAR,D,691
2,200001,SCAR,I,139
3,200001,SCBA,D,382
4,200001,SCCC,D,82


In [3]:
print("===== INFORMACIÓN GENERAL =====")
print("Filas y columnas:", df.shape)

print("\n===== TIPOS DE DATOS =====")
print(df.dtypes)

print("\n===== VALORES NULOS =====")
print(df.isna().sum())

print("\n===== FILAS DUPLICADAS =====")
print(df.duplicated().sum())

print("\n===== VALORES ÚNICOS POR COLUMNA =====")
print(df.nunique())

print("\n===== ESTADÍSTICAS DESCRIPTIVAS =====")
print(df.describe(include="all"))

===== INFORMACIÓN GENERAL =====
Filas y columnas: (16172, 4)

===== TIPOS DE DATOS =====
mes_id                     int64
aeropuerto_oaci              str
internacional_domestico      str
cnt_operaciones            int64
dtype: object

===== VALORES NULOS =====
mes_id                     0
aeropuerto_oaci            0
internacional_domestico    0
cnt_operaciones            0
dtype: int64

===== FILAS DUPLICADAS =====
0

===== VALORES ÚNICOS POR COLUMNA =====
mes_id                      321
aeropuerto_oaci              69
internacional_domestico       2
cnt_operaciones            2837
dtype: int64

===== ESTADÍSTICAS DESCRIPTIVAS =====
               mes_id aeropuerto_oaci internacional_domestico  cnt_operaciones
count    16172.000000           16172                   16172     16172.000000
unique            NaN              69                       2              NaN
top               NaN            SCAR                       D              NaN
freq              NaN             642    

In [4]:
print("===== TIPOS DE OPERACIÓN =====")
print(df["internacional_domestico"].value_counts())

print("\n===== OPERACIONES NEGATIVAS =====")
print((df["cnt_operaciones"] < 0).sum())

print("\n===== OPERACIONES EN CERO =====")
print((df["cnt_operaciones"] == 0).sum())

print("\n===== REGISTROS REPETIDOS POR CLAVE =====")
claves = ["mes_id", "aeropuerto_oaci", "internacional_domestico"]
print(df.duplicated(subset=claves).sum())

print("\n===== RANGO DE MESES =====")
print("Primer mes:", df["mes_id"].min())
print("Último mes:", df["mes_id"].max())

print("\n===== VALORES MÍNIMO Y MÁXIMO =====")
print("Mínimo:", df["cnt_operaciones"].min())
print("Máximo:", df["cnt_operaciones"].max())

===== TIPOS DE OPERACIÓN =====
internacional_domestico
D    12138
I     4034
Name: count, dtype: int64

===== OPERACIONES NEGATIVAS =====
0

===== OPERACIONES EN CERO =====
0

===== REGISTROS REPETIDOS POR CLAVE =====
0

===== RANGO DE MESES =====
Primer mes: 200001
Último mes: 202609

===== VALORES MÍNIMO Y MÁXIMO =====
Mínimo: 1
Máximo: 10722


In [5]:
print("===== VALIDACIÓN DE FECHAS =====")

fechas = pd.to_datetime(
    df["mes_id"].astype(str),
    format="%Y%m",
    errors="coerce"
)

print("Fechas inválidas:", fechas.isna().sum())

print("\n===== CÓDIGOS DE AEROPUERTO =====")
print("Códigos vacíos:", df["aeropuerto_oaci"].str.strip().eq("").sum())
print("Longitudes de códigos:")
print(df["aeropuerto_oaci"].str.len().value_counts())

print("\n===== CATEGORÍAS =====")
print(df["internacional_domestico"].unique())

===== VALIDACIÓN DE FECHAS =====
Fechas inválidas: 0

===== CÓDIGOS DE AEROPUERTO =====
Códigos vacíos: 0
Longitudes de códigos:
aeropuerto_oaci
4    16172
Name: count, dtype: int64

===== CATEGORÍAS =====
<ArrowStringArray>
['D', 'I']
Length: 2, dtype: str


## Informe de calidad de datos

### Descripción del conjunto de datos

Se utilizó el conjunto de datos "Operaciones de Aeronaves",
publicado por la Junta de Aeronáutica Civil de Chile.

El archivo contiene 16.172 registros y 4 columnas,
correspondientes a operaciones aéreas por mes, aeropuerto
y tipo de operación.

### Resultados del perfilamiento

- Valores nulos: 0.
- Filas duplicadas: 0.
- Duplicados por clave compuesta: 0.
- Fechas inválidas: 0.
- Operaciones negativas: 0.
- Operaciones en cero: 0.
- Códigos de aeropuerto vacíos: 0.
- Códigos de aeropuerto con longitud distinta de 4: 0.
- Categorías identificadas: D (doméstico) e I (internacional).
- Período: enero de 2000 a septiembre de 2026.

### Decisiones de calidad

No se eliminarán registros debido a que las validaciones
realizadas no identificaron valores nulos, duplicados
ni cantidades negativas.

Se conservará el archivo original sin modificaciones.
Las transformaciones se realizarán sobre una copia de
los datos y se guardarán en la carpeta data/processed.

### Conclusión

El conjunto de datos presenta una calidad adecuada según
las validaciones realizadas, por lo que puede utilizarse
para continuar con el análisis exploratorio y estadístico.

In [6]:
# Crear una copia para no modificar los datos originales
df_limpio = df.copy()

# Convertir el identificador del mes en fecha
df_limpio["fecha"] = pd.to_datetime(
    df_limpio["mes_id"].astype(str),
    format="%Y%m"
)

# Crear nuevas variables temporales
df_limpio["anio"] = df_limpio["fecha"].dt.year
df_limpio["mes"] = df_limpio["fecha"].dt.month

# Crear una descripción del tipo de operación
df_limpio["tipo_operacion"] = (
    df_limpio["internacional_domestico"].map({
        "D": "Doméstico",
        "I": "Internacional"
    })
)

# Mostrar los primeros registros
df_limpio.head()

,mes_id,aeropuerto_oaci,internacional_domestico,cnt_operaciones,fecha,anio,mes,tipo_operacion
0,200001,SCAP,D,6,2000-01-01,2000,1,Doméstico
1,200001,SCAR,D,691,2000-01-01,2000,1,Doméstico
2,200001,SCAR,I,139,2000-01-01,2000,1,Internacional
3,200001,SCBA,D,382,2000-01-01,2000,1,Doméstico
4,200001,SCCC,D,82,2000-01-01,2000,1,Doméstico


In [7]:
from pathlib import Path

# Localizar la carpeta del proyecto
carpeta_proyecto = ruta.resolve().parents[2]
carpeta_salida = carpeta_proyecto / "data" / "processed"

# Crear la carpeta si no existe
carpeta_salida.mkdir(parents=True, exist_ok=True)

# Verificar que no se perdieron registros
assert len(df_limpio) == len(df)
assert df_limpio["fecha"].notna().all()
assert df_limpio["tipo_operacion"].notna().all()

# Guardar en formato CSV
df_limpio.to_csv(
    carpeta_salida / "operaciones_procesadas.csv",
    index=False,
    encoding="utf-8-sig"
)

# Guardar en formato Parquet
df_limpio.to_parquet(
    carpeta_salida / "operaciones_procesadas.parquet",
    index=False
)

print("¡Archivos guardados correctamente!")
print("Registros:", len(df_limpio))
print("Columnas:", len(df_limpio.columns))
print("Carpeta:", carpeta_salida)

¡Archivos guardados correctamente!
Registros: 16172
Columnas: 8
Carpeta: C:\Users\catal\OneDrive\Escritorio\evaluacion_02_transportes\data\processed


In [8]:
# Agrupar la cantidad de operaciones por año
operaciones_anuales = (
    df_limpio.groupby("anio")["cnt_operaciones"]
    .sum()
    .reset_index()
)

# Ordenar cronológicamente
operaciones_anuales = operaciones_anuales.sort_values("anio")

print("===== OPERACIONES POR AÑO =====")
print(operaciones_anuales.to_string(index=False))

print("\n===== RESUMEN =====")
print("Total de operaciones:", df_limpio["cnt_operaciones"].sum())
print("Años analizados:", df_limpio["anio"].nunique())

===== OPERACIONES POR AÑO =====
 anio  cnt_operaciones
 2000           330720
 2001           312769
 2002           342309
 2003           353627
 2004           339849
 2005           354470
 2006           327726
 2007           338638
 2008           319240
 2009           312520
 2010           344320
 2011           387733
 2012           417820
 2013           445752
 2014           423833
 2015           417541
 2016           424996
 2017           442009
 2018           476266
 2019           505786
 2020           279989
 2021           362151
 2022           468362
 2023           513864
 2024           531226
 2025           524301
 2026           384437

===== RESUMEN =====
Total de operaciones: 10682254
Años analizados: 27


In [9]:
# Agrupar operaciones por aeropuerto
operaciones_aeropuerto = (
    df_limpio.groupby("aeropuerto_oaci")["cnt_operaciones"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

print("===== TOP 10 AEROPUERTOS =====")
print(operaciones_aeropuerto.head(10).to_string(index=False))

print("\n===== RESUMEN =====")
print("Aeropuertos analizados:", df_limpio["aeropuerto_oaci"].nunique())
print("Aeropuerto con más operaciones:",
      operaciones_aeropuerto.iloc[0]["aeropuerto_oaci"])

===== TOP 10 AEROPUERTOS =====
aeropuerto_oaci  cnt_operaciones
           SCEL          3132971
           SCTB          1284352
           SCIE           526012
           SCFA           491290
           SCDA           440760
           SCTE           423980
           SCSE           316396
           SCPF           309651
           SCRD           296260
           SCCF           273688

===== RESUMEN =====
Aeropuertos analizados: 69
Aeropuerto con más operaciones: SCEL


In [10]:
# Agrupar operaciones según su tipo
operaciones_tipo = (
    df_limpio.groupby("tipo_operacion")["cnt_operaciones"]
    .sum()
    .reset_index()
)

# Calcular porcentaje de participación
total = operaciones_tipo["cnt_operaciones"].sum()

operaciones_tipo["porcentaje"] = (
    operaciones_tipo["cnt_operaciones"] / total * 100
).round(2)

print("===== OPERACIONES POR TIPO =====")
print(operaciones_tipo.to_string(index=False))

print("\n===== TOTAL GENERAL =====")
print(f"{total:,}")

===== OPERACIONES POR TIPO =====
tipo_operacion  cnt_operaciones  porcentaje
     Doméstico          9251397       86.61
 Internacional          1430857       13.39

===== TOTAL GENERAL =====
10,682,254


In [11]:
# Utilizar solo años completos para evitar sesgos
anios_completos = df_limpio[df_limpio["anio"] < 2026]

# Agrupar operaciones por año y mes
operaciones_anio_mes = (
    anios_completos.groupby(["anio", "mes"])["cnt_operaciones"]
    .sum()
    .reset_index()
)

# Calcular el promedio de operaciones de cada mes
operaciones_mensuales = (
    operaciones_anio_mes.groupby("mes")["cnt_operaciones"]
    .mean()
    .round(2)
    .reset_index(name="promedio_operaciones")
)

print("===== PROMEDIO DE OPERACIONES POR MES =====")
print(operaciones_mensuales.to_string(index=False))

print("\n===== MES CON MAYOR PROMEDIO =====")
print(operaciones_mensuales.loc[
    operaciones_mensuales["promedio_operaciones"].idxmax()
].to_string())

===== PROMEDIO DE OPERACIONES POR MES =====
 mes  promedio_operaciones
   1              40433.38
   2              35567.04
   3              35041.04
   4              29465.85
   5              28525.73
   6              26898.81
   7              30259.88
   8              29872.73
   9              30427.19
  10              35029.73
  11              36444.77
  12              38103.73

===== MES CON MAYOR PROMEDIO =====
mes                         1.00
promedio_operaciones    40433.38
